In [0]:
%run ../setup/config

In [0]:
%run ../setup/utils

In [0]:
movies_metadata_df = spark.read.format("delta").load(f"{silver_folder_path}/movies_metadata")
ratings_df = spark.read.format("delta").load(f"{silver_folder_path}/ratings")
links_df = spark.read.format("delta").load(f"{silver_folder_path}/links")
genres_df = spark.read.format("delta").load(f"{silver_folder_path}/movie_genres")

movies_metadata_df.printSchema()
ratings_df.printSchema()
links_df.printSchema()
genres_df.printSchema()

In [0]:
from pyspark.sql import functions as F

final_movies_df = (
  ratings_df
    .join(links_df, links_df.movie_id == ratings_df.movie_id, "inner")
    .join(movies_metadata_df, links_df.tmbd_id == movies_metadata_df.id, "inner")
    .join(genres_df, movies_metadata_df.id == genres_df.id, "inner")
    .groupBy(movies_metadata_df.id, "title", "genres_id", "genres_name")
    .agg(
        F.avg("rating").alias("average_rating"),
        F.count("user_id").alias("number_of_ratings"),
    )
    .filter("number_of_ratings > 100")
    .orderBy(F.col("average_rating").desc())
)

display(final_movies_df)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F

w = Window.partitionBy("genres_name").orderBy(F.col("average_rating").desc())
ranked_df = (
  final_movies_df
    .withColumn("rank_in_genre", F.rank().over(w))
)
display(
  ranked_df
    .filter("rank_in_genre <= 3")
    .orderBy("genres_name", "rank_in_genre")
)